In [82]:
import pandas as pd
import os

In [83]:
df = pd.read_csv("Price_monitoring_table1.csv")

In [84]:
df["Advertised_price_INR"] = pd.to_numeric(df["Advertised_price_INR"], errors="coerce")
df["promotion"] = pd.to_numeric(df["promotion"], errors="coerce")
df["LPP"] = pd.to_numeric(df["LPP"], errors="coerce")

In [85]:
df["Violation"] = 0

Rule 1
Q2 + promotion exists
Advertised < promotion → violation

Rule 2
Q2 + promotion missing
Advertised < LPP → violation

Rule 3
Not Q2
Advertised < LPP → violation

In [86]:
# Q2 with promotion

mask_rule1 = ((df["season"] == "Q2") &(df["promotion"].notna()) &(df["Advertised_price_INR"] < df["promotion"]))

df.loc[mask_rule1, "Violation"] = 1

In [87]:
# Q2 without promotion

mask_rule2 = ((df["season"] == "Q2") &(df["promotion"].isna()) &(df["Advertised_price_INR"] < df["LPP"]))

df.loc[mask_rule2, "Violation"] = 1

In [88]:
# Not Q2

mask_rule3 = ((df["season"] != "Q2") &(df["Advertised_price_INR"] < df["LPP"]))

df.loc[mask_rule3, "Violation"] = 1

In [89]:
df_violation = df[df["Violation"] == 1].copy()

In [91]:
# Creating letter Folders

In [92]:
os.makedirs("letters/warning", exist_ok=True)
os.makedirs("letters/violation", exist_ok=True)
os.makedirs("letters/suspension", exist_ok=True)

In [93]:
df_violation["violation_count"] = (df_violation.groupby(["sku","Homologated_sellers_name"]).cumcount() + 1)

In [94]:
def get_type(x):

    if x == 1:
        return "warning"

    elif x == 2:
        return "violation"

    else:
        return "suspension"


df_violation["violation_type"] = df_violation["violation_count"].apply(get_type)

In [95]:
# Warning Template

warning_template = """
Subject: Warning Notice – MAP Pricing Non-Compliance

Dear {Homologated_sellers_name},

We have identified a pricing discrepancy for the following product listed on {Marketplace}. The advertised price is currently below the permitted price level.

Product Details

SKU: {sku}
Violation Date: {Violation date}
Product Line (PL): {PL}
Category: {category}
Sub Category: {subcategory}
Region: {region}
Marketplace: {Marketplace}

Pricing Information

MAP Price: {MAP}
LPP: {LPP}
Season: {season}
Promotional Price: {promotion}
Advertised Price: {adv_price}
Advertised Price (INR): {Advertised_price_INR}

This letter serves as a formal warning regarding non-compliance with the pricing policy.

Please correct the pricing immediately.

Sincerely,
Pricing Compliance Team
"""

In [96]:
# Violation Template

violation_template = """
Subject: Violation Notice – MAP Pricing Policy Breach

Dear {Homologated_sellers_name},

This notice serves as a formal violation notice regarding non-compliance with our pricing policy.

Product Details

SKU: {sku}
Violation Date: {Violation date}
Product Line (PL): {PL}
Category: {category}
Sub Category: {subcategory}
Region: {region}
Marketplace: {Marketplace}

Pricing Information

MAP Price: {MAP}
LPP: {LPP}
Season: {season}
Promotional Price: {promotion}
Advertised Price: {adv_price}
Advertised Price (INR): {Advertised_price_INR}

This represents a second instance of pricing non-compliance.

Immediate correction is required.

Sincerely,
Pricing Compliance Team
"""

In [97]:
# Suspension Template

suspension_template = """
Subject: Suspension Notice – Category Sales Suspension Due to Pricing Violations

Dear {Homologated_sellers_name},

Following repeated violations of the pricing policy, we regret to inform you that sales activity for the affected category will be suspended on the specified marketplace.

The violation details are listed below:

Product Details

SKU: {sku}
Violation Date: {Violation date}
Product Line (PL): {PL}
Category: {category}
Sub Category: {subcategory}
Region: {region}
Marketplace: {Marketplace}

Pricing Information

MAP Price: {MAP}
LPP: {LPP}
Season: {season}
Promotional Price: {promotion}
Advertised Price: {adv_price}
Advertised Price (INR): {Advertised_price_INR}

Due to repeated non-compliance with the pricing policy, sales of products under the category "{category}" on the marketplace "{Marketplace}" are hereby suspended until further notice.

To restore selling privileges, you must ensure full compliance with the pricing policy and contact the compliance team for review.

Sincerely,
Pricing Compliance Team
"""

In [1]:
import re

def clean_filename(text):
    text = str(text)
    text = re.sub(r'[\\/*?:"<>|]', "", text)  # remove unwanted characters
    text = text.replace(" ", "_")             # replace spaces
    return text

In [99]:
for _, row in df_violation.iterrows():

    if row["violation_type"] == "warning":
        template = warning_template
        folder = "letters/warning"

    elif row["violation_type"] == "violation":
        template = violation_template
        folder = "letters/violation"

    else:
        template = suspension_template
        folder = "letters/suspension"

    letter = template.format(**row.to_dict())

    seller = clean_filename(row["Homologated_sellers_name"])
    sku = clean_filename(row["sku"])

    filename = f"{folder}/{seller}_{sku}.txt"

    with open(filename, "w") as f:
        f.write(letter)

In [100]:
df_violation = df_violation[[
    "sku",
    "Violation date",
    "PL",
    "category",
    "subcategory",
    "MAP",
    "LPP",
    "season",
    "promotion",
    "Marketplace",
    "Seller_name",
    "Homologated_sellers_name",
    "region",
    "adv_price",
    "Advertised_price_INR"
]]

df_violation.to_csv("Violation_table1.csv", index=False)